In [3]:
import numpy as np
import pandas as pd
import pickle
import tensorflow as tf
from tensorflow.keras.models import load_model


In [7]:
import pickle

# load artifacts
with open("onehot_encoder_geo.pkl", "rb") as f:
    onehot_geo = pickle.load(f)

with open("label_encoder_gender.pkl", "rb") as f:
    label_enc_gender = pickle.load(f)

with open("scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

from tensorflow.keras.models import load_model
model = load_model("model.h5")


In [5]:
## Example input data
input_data ={
    'CreditScore': 600,
    'Geography': 'France',
    'Gender':'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember':1,
    'EstimatedSalary':50000
}

In [9]:
import pandas as pd

geo_encoded = onehot_geo.transform(pd.DataFrame({'Geography': [input_data['Geography']]}))
# only convert if the result is sparse
if hasattr(geo_encoded, 'toarray'):
    geo_encoded = geo_encoded.toarray()

geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_geo.get_feature_names_out(['Geography'])
)
geo_encoded_df


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [10]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [11]:
## Encode categorical variables
input_df['Gender']=label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [12]:
## concatination one hot encoded 
input_df=pd.concat([input_df.drop("Geography",axis=1),geo_encoded_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [13]:
## Scaling the input data
input_scaled=scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [14]:
## PRedict churn
prediction=model.predict(input_scaled)
prediction

1/1 [==============================] - 7s 7s/step


array([[0.07207616]], dtype=float32)

In [15]:
prediction_proba = prediction[0][0]

In [16]:
prediction_proba

0.072076164

In [17]:
if prediction_proba > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.
